# 👗 Kleding Classifier — Bulk Upload

Dit notebook herkent kledingstukken in bulk en sorteert ze in categorieën:

| Categorie | Voorbeelden |
|---|---|
| 👕 T-shirt / Top | t-shirt, sweatshirt, bh, bikini |
| 👖 Broek / Jeans | jeans, broek |
| 🧥 Jas / Vest | jas, vest, cardigan, trenchcoat |
| 👗 Jurk / Rok | jurk, rok, minirok |
| 👟 Schoenen | sneakers, laarzen, sandalen |
| 👜 Accessoires | tas, portemonnee, riem, sjaal, horloge |
| ❓ Overige | alles wat geen kleding is |

---
**Hoe te gebruiken:**
1. Zet al je kledingfoto's in een map genaamd `kleding` (naast dit notebook)
2. Voer alle cellen uit van boven naar beneden met Shift+Enter
3. Bekijk het resultaat als overzichtsgrid

## Stap 1 — Importeren & model laden

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import json
import os
from io import BytesIO
from pathlib import Path
from IPython.display import display, HTML

print('⏳ Model laden...')
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.eval()

# Labels laden
try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    labels = json.loads(requests.get(
        'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json',
        headers=headers, timeout=5
    ).text)
except:
    labels = models.MobileNet_V2_Weights.IMAGENET1K_V1.meta['categories']

print(f'✅ Model geladen! ({len(labels)} klassen)')

## Stap 2 — Categoriedefinities & classificatielogica

In [ ]:
# Kleding-indices per categorie (ImageNet klasse-nummers)
CATEGORIEEN = {
    '👕 T-shirt / Top':    [459, 610, 841, 842, 445, 639],      # bra, T-shirt, sweatshirt, swimsuit, bikini, tank suit
    '👖 Broek / Jeans':    [608],                                # jeans
    '🧥 Jas / Vest':       [474, 568, 617, 834, 869, 465, 887], # cardigan, fur coat, lab coat, suit, trench coat, bulletproof vest, vestment
    '👗 Jurk / Rok':       [601, 655, 689],                     # hoop skirt, miniskirt, overskirt
    '👟 Schoenen':         [514, 630, 770, 774, 806],           # cowboy boot, slip-on shoe, running shoe, sandal, sock
    '👜 Accessoires':      [451, 457, 531, 636, 728, 747, 748, 785, 824, 893, 906],  # bolo tie, bow tie, watch, mail bag, plastic bag, punching bag, purse, seat belt, scarf, wallet, Windsor tie
}

KLEUREN = {
    '👕 T-shirt / Top':  '#e8a87c',
    '👖 Broek / Jeans':  '#6b8cba',
    '🧥 Jas / Vest':     '#7daa7d',
    '👗 Jurk / Rok':     '#c47db5',
    '👟 Schoenen':       '#e07070',
    '👜 Accessoires':    '#d4a84b',
    '❓ Overige':        '#aaaaaa',
}

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def classificeer(img: Image.Image, drempel: float = 0.05):
    """
    Classificeert een kledingafbeelding.
    drempel: minimale kans om als kleding te worden herkend (standaard 5%)
    """
    tensor = preprocess(img.convert('RGB')).unsqueeze(0)
    with torch.no_grad():
        output = model(tensor)
    probs = torch.nn.functional.softmax(output[0], dim=0)

    # Kansen per categorie optellen
    scores = {}
    for naam, indices in CATEGORIEEN.items():
        scores[naam] = float(sum(probs[i] for i in indices))

    beste = max(scores, key=scores.get)
    beste_kans = scores[beste]

    if beste_kans < drempel:
        return {'label': '❓ Overige', 'kans': 1 - sum(scores.values()), 'scores': scores}

    return {'label': beste, 'kans': beste_kans, 'scores': scores}

print('✅ Categorieën gedefinieerd:')
for cat in CATEGORIEEN:
    print(f'   {cat}')
print('   ❓ Overige')

## Stap 3 — Bulk classificatie

Zet je foto's in een map genaamd **`kleding`** naast dit notebook, en run de cel hieronder.

In [ ]:
# -------------------------------------------------------
# 🔧 AANPASSEN: verander naar jouw mapnaam indien nodig
# -------------------------------------------------------
MAP = 'kleding'
EXTENSIES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

if not os.path.isdir(MAP):
    print(f'⚠️  Map "{MAP}" niet gevonden!')
    print(f'   Maak een map aan genaamd "{MAP}" naast dit notebook')
    print(f'   en zet daar je kledingfoto\'s in.')
else:
    bestanden = sorted([
        f for f in Path(MAP).iterdir()
        if f.suffix.lower() in EXTENSIES
    ])

    if not bestanden:
        print(f'⚠️  Geen afbeeldingen gevonden in "{MAP}"')
    else:
        print(f'📁 {len(bestanden)} afbeeldingen gevonden, bezig met classificeren...\n')

        resultaten = []
        for i, pad in enumerate(bestanden, 1):
            try:
                img = Image.open(pad)
                r = classificeer(img)
                resultaten.append({'pad': pad, 'img': img.copy(), **r})
                print(f'  [{i:>3}/{len(bestanden)}] {pad.name:35s} → {r["label"]:22s} ({r["kans"]*100:.0f}%)')
            except Exception as e:
                print(f'  ❌ {pad.name}: {e}')

        print(f'\n✅ Klaar! {len(resultaten)} afbeeldingen geclassificeerd.')

## Stap 4 — Resultaten bekijken per categorie

In [ ]:
if not resultaten:
    print('Geen resultaten om te tonen — run Stap 3 eerst.')
else:
    # Groepeer per categorie
    from collections import defaultdict
    groepen = defaultdict(list)
    for r in resultaten:
        groepen[r['label']].append(r)

    print(f'📊 Samenvatting ({len(resultaten)} foto\'s):')
    for label, items in sorted(groepen.items()):
        kleur = KLEUREN.get(label, '#aaa')
        print(f'   {label}: {len(items)}x')

    print('\n' + '='*60)

    # Toon per categorie een grid
    for label, items in sorted(groepen.items()):
        kleur = KLEUREN.get(label, '#aaaaaa')
        print(f'\n{label} — {len(items)} foto(\'s)')

        cols = min(4, len(items))
        rows = (len(items) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows))

        # Zorg altijd voor een lijst van assen
        if len(items) == 1:
            axes = [axes]
        elif rows == 1:
            axes = list(axes)
        else:
            axes = axes.flatten().tolist()

        for ax, item in zip(axes, items):
            ax.imshow(item['img'])
            ax.axis('off')
            ax.set_title(
                f"{item['pad'].name}\n{item['kans']*100:.0f}% zekerheid",
                fontsize=9, color=kleur, fontweight='bold'
            )
            for spine in ax.spines.values():
                spine.set_edgecolor(kleur)
                spine.set_linewidth(3)
                spine.set_visible(True)

        for ax in axes[len(items):]:
            ax.set_visible(False)

        fig.suptitle(label, fontsize=14, fontweight='bold', color=kleur, y=1.01)
        plt.tight_layout()
        plt.show()

## Stap 5 — Scoreoverzicht per foto (grafiek)

In [ ]:
# Kies een specifieke foto om de scores van alle categorieën te zien
# Verander de index (0 = eerste foto, 1 = tweede, etc.)
INDEX = 0

if not resultaten:
    print('Geen resultaten — run Stap 3 eerst.')
elif INDEX >= len(resultaten):
    print(f'Index {INDEX} bestaat niet, er zijn maar {len(resultaten)} foto\'s.')
else:
    item = resultaten[INDEX]
    scores = item['scores']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Afbeelding
    ax1.imshow(item['img'])
    ax1.axis('off')
    kleur = KLEUREN.get(item['label'], '#aaa')
    ax1.set_title(
        f"{item['label']}\n{item['kans']*100:.1f}% zekerheid",
        color=kleur, fontweight='bold', fontsize=13
    )

    # Staafdiagram alle categorieën
    namen  = list(scores.keys())
    kansen = [scores[n]*100 for n in namen]
    kleuren = [KLEUREN.get(n, '#aaa') for n in namen]
    bars = ax2.barh(namen, kansen, color=kleuren)
    ax2.set_xlabel('Kans (%)')
    ax2.set_title(f'Scores voor: {item["pad"].name}', fontsize=12)
    for bar, k in zip(bars, kansen):
        ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{k:.1f}%', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

---
## 💡 Tips

**Drempel aanpassen** (als te veel items als 'Overige' verschijnen):
```python
r = classificeer(img, drempel=0.02)  # minder streng
r = classificeer(img, drempel=0.10)  # strenger
```

**Specifieke foto via INDEX bekijken** in Stap 5:
```python
INDEX = 3  # bekijk de 4e foto
```

**Volgende stap:** resultaten exporteren naar CSV of JSON voor gebruik in KledingHelper.